# Mixed flocks in open space: every simulation, table and figure of the paper

Companion notebook to *Segregation with cohesion in mixed flocks from interfering perceptual cues*, by Mayank Chaurasiya, Ayush Agrawal, Chirayush Mohanty and Laxmidhar Behera.

**The model.** Agents of $G = 3$ kinds move at constant speed in an open plane. Each agent perceives $n = 6$ partners, redrawn at every step. It holds its own heading, the partners' headings and the partners' bearings in one perceptual state and draws its next heading from that state (Born rule). Identity enters in two ways:

1. **Relational partners.** Partners are sampled among all visible individuals in proportion to their similarity to the agent, regardless of distance.
2. **Identity phase.** The bearing of a partner of another kind enters the perceptual state with phase $\pi$ (a minus sign).

**How to run.** Change values only in Section 1, then *Kernel → Restart & Run All*. Set `QUICK_TEST = True` first to check that every cell runs (a few minutes), then set it back to `False`.

**Time.** One run of 5,000 steps takes about 25 s for 240 agents, 80 s for 480 and 4 min for 960, on one core. The whole notebook is about 20 core-hours (about 2 hours on 10 cores); Section 7 is half of that. Every run is saved as soon as it finishes, under a key made from all of its parameters, so the notebook can be stopped and restarted and only what is missing is computed. To shorten Section 7, remove the entry `960` from `LIMIT_PLAN`.

**Where each result of the paper comes from** (all files go to `output/`)

| Section | Paper | Output |
|---|---|---|
| 5 | Table 2, Figures 2 and 3 | `table1_cases.csv`, `fig2_flocks.png`, `fig3_over_time.png` |
| 6 | Table 3 | `table2_rules.csv`, `table2_rules_tests.csv` |
| 7 | Figure 4, Table A6 | `fig4_sight_range.png`, `table_size_limit.csv` |
| 8 | Figure 5 | `fig5_attention.png`, `table_attention.csv` |
| 9 | Figure 6 | `fig6_noise.png`, `table_noise.csv`, `table_noise_map.csv` |
| 10 | Figure 7 | `fig7_sensitivity.png`, `table_sensitivity.csv` |
| 11 | Table 4, large flock | `table_cues.csv`, `table_large_flock.csv` |
| 12 | Appendix E, Figure 8 | `fig8_perception_strength.png`, `table_perception_strength.csv` |
| 13 | Figure 9, long runs | `table_long_runs.csv`, `fig9_long_runs.png` |
| 14 | Figure 1, Appendix D | `fig1_framework.png`, `table_worked_example.csv` |
| 15 | Section 3.4 (Vicsek limit) | `table_vicsek_limit.csv` |
| 16 | every number quoted in the text | `numbers_for_manuscript.txt` |
| 17 | all data in one file | `all_runs.csv`, `time_series.npz`, `final_positions_fig2.npz`, `paper_data.zip` |
| 18 | supplementary video | `video_three_cases.mp4` |

The scripts in `checks/` give the few numbers that are not in this notebook (see `README.md`).


In [ ]:
import os, json, time, hashlib, pickle, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Wedge
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.special import i0e, i1e
from scipy.stats import mannwhitneyu
from IPython.display import Video, Image, display

INK, INK2, GRID = "#0b0b0b", "#52514e", "#d9d8d2"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "legend.frameon": False,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "axes.labelcolor": INK})
KIND_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#e87ba4", "#eda100"]
PI = float(np.pi)

# A case says: which share of the partners are the nearest ones (the rest are sampled from the whole view),
# whether similarity is used in the sampling, the phase of the bearing of a partner of another kind,
# and the decision rule.
CASES = {"no_identity": dict(near=0.0, similarity=False, phase=0.0, rule="born"),
         "relational":  dict(near=0.0, similarity=True,  phase=0.0, rule="born"),
         "full":        dict(near=0.0, similarity=True,  phase=PI,  rule="born"),
         "nearest":     dict(near=1.0, similarity=False, phase=PI,  rule="born"),
         "nearest_no_identity": dict(near=1.0, similarity=False, phase=0.0, rule="born"),
         "half_nearest": dict(near=0.5, similarity=True,  phase=PI,  rule="born"),
         "decohered":   dict(near=0.0, similarity=True,  phase=PI,  rule="decohered"),
         "vector":      dict(near=0.0, similarity=True,  phase=PI,  rule="vector"),
         "vector_zero": dict(near=0.0, similarity=True, phase=0.0, rule="vector", other_bearing=0.0),   # other kinds' bearings ignored
         "peak":        dict(near=0.0, similarity=True,  phase=PI,  rule="peak"),       # always the most likely heading, no draw
         "peak_noise":  dict(near=0.0, similarity=True,  phase=PI,  rule="peak_noise"), # most likely heading + noise of fixed width
         "peak_width":  dict(near=0.0, similarity=True,  phase=PI,  rule="peak_width"), # most likely heading + noise as wide as the draw
         "peak_noid":   dict(near=0.0, similarity=False, phase=0.0, rule="peak"),        # the same two rules without identity
         "peak_noise_noid": dict(near=0.0, similarity=False, phase=0.0, rule="peak_noise"),
         "mean_width":  dict(near=0.0, similarity=True,  phase=PI,  rule="mean_width"), # expected heading + noise as wide as the draw
         "mean_width_noid": dict(near=0.0, similarity=False, phase=0.0, rule="mean_width"),
         "ignore_other": dict(near=0.0, similarity=True, phase=0.0, rule="born", other_bearing=0.0)}  # model, other kinds' bearings given zero weight
LABEL = {"no_identity": "no identity", "relational": "relational partners, no phase", "full": "full model",
         "nearest": "nearest partners", "nearest_no_identity": "nearest partners, no identity",
         "half_nearest": "half nearest, half relational",
         "decohered": "probabilities added (no interference)", "vector": "signed vector sum (classical)",
         "vector_zero": "vector sum, other kinds' bearings ignored",
         "peak": "most likely heading taken (no draw)",
         "peak_noise": "most likely heading + noise of fixed width",
         "peak_width": "most likely heading + noise as wide as the draw",
         "peak_noid": "most likely heading, no identity", "peak_noise_noid": "most likely heading + fixed noise, no identity",
         "mean_width": "expected heading + noise as wide as the draw", "mean_width_noid": "expected heading + matched noise, no identity",
         "ignore_other": "relational partners, other kinds' bearings zero weight"}
TABLE1 = ["no_identity", "relational", "ignore_other", "full", "nearest", "nearest_no_identity", "half_nearest"]
SHOW = ["no_identity", "relational", "full"]            # drawn in the figures and the video
COLOR = {"no_identity": "#8a8a86", "relational": "#eda100", "full": "#4a3aa7"}
DASH = {"no_identity": (0, (4, 2)), "relational": "-", "full": "-"}
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
print("numpy", np.__version__, "| pandas", pd.__version__, "| matplotlib", matplotlib.__version__, "| CPU cores:", os.cpu_count())

## 1. Parameters — change values here only

| Symbol | Meaning | Value |
|---|---|---|
| $N$ | number of agents | 240 |
| $G$ | number of kinds (equal numbers) | 3 |
| $v_0$ | speed, body lengths per step | 0.5 |
| $R$ | range of sight, body lengths | 25 |
| — | field of view | 270° |
| $n$ | partners perceived by an agent | 6 |
| $w$ | weight of a partner's heading ($1-w$ for its bearing) | 0.8 |
| $\lambda$ | sharpness of perception | 50 |
| $\mu$ | memory of the perceptual state (20 steps) | 0.95 |
| $\delta_0$ | share of attention to other kinds | 0.3 |

In [ ]:
QUICK_TEST = False          # True: tiny runs to check that every cell works (about two minutes)
N_JOBS     = 4             # -1 = use all CPU cores; or a number, e.g. 32

BASE = dict(N=240, G=3, v0=0.5, r0=1.0, R=25.0, fov_deg=270.0, n=6, w=0.8, lam=50.0, mu=0.95, delta0=0.3,
            n_dir=120, density=0.25, passes=20)
T_RUN  = 5000               # steps per run
EVERY  = 10                 # record the measures every EVERY steps
C_ONE  = 0.9                # a run ends as one group if C > C_ONE
CONTACT = 3.0               # body lengths: agents closer than this are "in contact" (second, stricter grouping)

MAIN_SEEDS   = list(range(3000, 3030))             # Tables 2 and 3 of the paper (30 runs per case)
SWEEP_SEEDS  = list(range(2000, 2020))             # delta0 sweep (30 runs per value)
SENS_SEEDS   = list(range(2000, 2010))             # noise and sensitivity sweeps (20 runs per value)
SWEEP_DELTAS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, "random"]   # "random": similarity ignored, delta0 = (G-1)/G

# Section 7: flock width / sight range.  For each flock size: the ratios tested and the number of runs.
LIMIT_RATIOS = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0]
LIMIT_PLAN   = {240: (LIMIT_RATIOS, 20), 480: (LIMIT_RATIOS, 10), 960: ([2.0, 2.5, 3.0, 3.5, 4.5], 6)}
LIMIT_SEED0  = 4000

NOISE_LAMS = [5.0, 10.0, 20.0, 50.0, 100.0]        # Section 9
SENS = {"phase": [0.0, PI / 4, PI / 2, 3 * PI / 4, PI],      # Section 10; each value replaces the one in BASE / the full model
        "n":     [2, 3, 4, 6, 9, 12],
        "w":     [0.5, 0.65, 0.8, 0.9, 0.95],
        "mu":    [0.0, 0.8, 0.95, 0.99],
        "G":     [2, 3, 4, 5]}
LONG_T, LONG_SEEDS = 15000, list(range(3000, 3010))
BIG_N, BIG_RATIOS, BIG_T, BIG_SEEDS = 960, [3.0, 4.5], 15000, [4000, 4001]   # Section 11: large flock followed for longer
SHOW_SEED    = 3000
VIDEO_EVERY, VIDEO_FPS = 10, 25

if QUICK_TEST:
    BASE["N"] = 60; T_RUN = 150
    MAIN_SEEDS = MAIN_SEEDS[:3]; SWEEP_SEEDS = SWEEP_SEEDS[:2]; SENS_SEEDS = SENS_SEEDS[:2]
    LIMIT_PLAN = {60: ([2.0, 4.0], 2), 120: ([2.0, 4.0], 2)}
    NOISE_LAMS = [10.0, 50.0]; SENS = {"phase": [PI / 2, PI], "n": [3, 6], "w": [0.8], "mu": [0.0, 0.95], "G": [2, 3]}
    LONG_T = 300; LONG_SEEDS = LONG_SEEDS[:2]; VIDEO_EVERY = 5
    BIG_N, BIG_RATIOS, BIG_T, BIG_SEEDS = 120, [2.0], 300, [4000]

OUT = "output"              # all tables, figures and saved runs go here
os.makedirs(os.path.join(OUT, "cache"), exist_ok=True)
G = BASE["G"]
print(f"N = {BASE['N']}, G = {G}, T = {T_RUN}, main runs per case = {len(MAIN_SEEDS)}, quick test = {QUICK_TEST}")

## 2. The model

**Perceived states.** A perceived direction $d$ is the state $\langle\theta|d\rangle = \exp[\tfrac{\lambda}{2}(\cos(\theta-d)-1)]$ over the possible headings $\theta$.

**Percept.** $|p_i\rangle \propto |S_i\rangle + \sum_{j\in\mathcal N_i}\big(\sqrt{w}\,|H_i^j\rangle + \sqrt{1-w}\;e^{i\varphi_{ij}}|B_i^j\rangle\big)$: own heading, partners' headings, partners' bearings. The identity phase is $\varphi_{ij} = 0$ for a partner of the same kind and $\varphi$ (normally $\pi$) for a partner of another kind.

**Perceptual state.** $|P_i\rangle \leftarrow \sqrt{\mu}\,|P_i\rangle + \sqrt{1-\mu}\,|p_i\rangle$, normalised.

**Decision.** The heading is drawn with probability $|\langle\theta|P_i\rangle|^2$ on a grid of 120 headings, plus a uniform jitter inside the grid cell. The agent moves by $v_0$ along it.

**Partners** are redrawn at every step. *Sampled:* every visible $j$ draws a waiting time $E_j\sim\mathrm{Exp}(1)$ and the smallest $E_j/S_{ij}$ are chosen; this is weighted sampling without replacement with weights $S_{ij}$ (1 for the same kind, $s=\delta_0/[(G-1)(1-\delta_0)]$ otherwise; all 1 when similarity is ignored). *Nearest:* the nearest visible individuals.

**Bodies.** After the move, any two agents closer than one body length are moved apart by half their overlap each; this is repeated until no overlap is left (at most `passes` times).

**Other decision rules**, used only in Section 6, with the same partners and the same perceived states:
- *probabilities added:* $\mathcal P_i(\theta)\propto\sum_m a_m^2\,|\langle\theta|m\rangle|^2$ (the perceptual state with its coherences removed); memory mixes the probabilities.
- *signed vector sum:* the heading is the direction of $\sum_m a_m\,\hat e(d_m)$, with the same signs, plus von Mises noise of concentration $\lambda$ (the noise of the model when all perceived directions agree); memory mixes the vectors.
- *most likely heading (`peak`):* the model's perceptual state, but the agent always takes the heading of largest probability.
- *most likely heading + fixed noise (`peak_noise`):* that heading plus von Mises noise of concentration $\lambda$.
- *most likely heading + matched noise (`peak_width`):* that heading plus von Mises noise whose width matches the spread of $|\langle\theta|P_i\rangle|^2$ (concentration from its mean resultant length, Best–Fisher approximation), per agent and step.
- *expected heading + matched noise (`mean_width`):* the same noise, centred on the expected (mean) heading of $|\langle\theta|P_i\rangle|^2$ instead of its peak.

**Zero weight (`ignore_other`)**, used in Sections 5 and 11: the model with the bearings of partners of other kinds given weight 0 instead of a minus sign.


In [ ]:
def similarity_s(G, delta0):
    # similarity of another kind, from the share of attention delta0 given to other kinds
    return delta0 / ((G - 1) * (1 - delta0))

def start(p, seed):
    # disordered, fully mixed start: positions in a disc, random headings, kinds shuffled
    rng = np.random.default_rng(seed)
    N = p["N"]
    radius = np.sqrt(N / (np.pi * p["density"]))
    r, a = radius * np.sqrt(rng.random(N)), 2 * np.pi * rng.random(N)
    X = np.c_[r * np.cos(a), r * np.sin(a)]
    theta = 2 * np.pi * rng.random(N)
    kind = rng.permutation(np.arange(N) % p["G"])
    return X, theta, kind, rng

def start_width(N, density=None):
    # diameter of the starting disc
    return 2 * np.sqrt(N / (np.pi * (BASE["density"] if density is None else density)))

def step(X, theta, mem, kind, S, case, rng, grid, p):
    # One time step for all agents. X: positions, theta: headings, mem: perceptual states.
    dirs, cos_d, sin_d = grid
    N, n = len(theta), p["n"]

    # 1. See. Only positions relative to the agent are used.
    dx = X[None, :, 0] - X[:, None, 0]
    dy = X[None, :, 1] - X[:, None, 1]
    dist = np.sqrt(dx * dx + dy * dy)
    np.fill_diagonal(dist, np.inf)
    cos_half = np.cos(np.deg2rad(p["fov_deg"]) / 2)
    with np.errstate(invalid="ignore"):
        visible = (dist <= p["R"]) & (dx * np.cos(theta)[:, None] + dy * np.sin(theta)[:, None] >= cos_half * dist)

    # 2. Choose n partners: k_near nearest ones, the others sampled from the rest of the view.
    k_near = int(round(case["near"] * n))
    rows = np.arange(N)[:, None]
    chosen, found = [], []
    free = visible
    if k_near:
        score = np.where(visible, -dist, -np.inf)
        idx = np.argpartition(-score, k_near - 1, axis=1)[:, :k_near]
        chosen.append(idx); found.append(np.isfinite(np.take_along_axis(score, idx, 1)))
        free = visible.copy(); free[rows, idx] = False
    if n - k_near:
        with np.errstate(divide="ignore"):
            score = np.where(free, -rng.standard_exponential((N, N)) / S, -np.inf)
        idx = np.argpartition(-score, n - k_near - 1, axis=1)[:, :n - k_near]
        chosen.append(idx); found.append(np.isfinite(np.take_along_axis(score, idx, 1)))
    partners = np.concatenate(chosen, 1)
    found = np.concatenate(found, 1)                                   # False where fewer partners are available

    # 3. Perceived directions and their amplitudes: own heading, partners' headings, partners' bearings.
    bearing = np.arctan2(dy[rows, partners], dx[rows, partners])
    direction = np.c_[theta, theta[partners], bearing]
    amplitude = np.c_[np.ones(N), np.sqrt(p["w"]) * found, np.sqrt(1 - p["w"]) * found]
    other = kind[partners] != kind[:, None]
    phase = case["phase"]
    if "other_bearing" in case:                                        # control: weight of the bearings of other kinds
        amplitude[:, 1 + n:] *= np.where(other, case["other_bearing"], 1.0)
    if abs(phase - PI) < 1e-12:                                        # phase pi: a minus sign
        sign = np.where(other, -1.0, 1.0)
        on = case.get("phase_on", "bearings")                          # controls can put the phase on the headings instead
        if on in ("bearings", "both"):
            amplitude[:, 1 + n:] *= sign
        if on in ("headings", "both"):
            amplitude[:, 1:1 + n] *= sign
    elif phase != 0.0:                                                 # any other phase: complex amplitudes
        amplitude = amplitude.astype(complex)
        amplitude[:, 1 + n:] *= np.where(other, np.exp(1j * phase), 1.0)
    amplitude /= np.linalg.norm(amplitude, axis=1, keepdims=True)

    # 4. Decision.
    if case["rule"] == "vector":                                       # classical: signed vector sum + noise
        v = np.c_[(amplitude * np.cos(direction)).sum(1), (amplitude * np.sin(direction)).sum(1)]
        mem = v if mem is None else p["mu"] * mem + (1 - p["mu"]) * v
        theta = rng.vonmises(np.arctan2(mem[:, 1], mem[:, 0]), p["lam"]) % (2 * np.pi)
    else:
        half = 0.5 * p["lam"]
        states = np.stack([np.cos(direction), np.sin(direction)], axis=2) @ (half * np.stack([cos_d, sin_d]))
        states -= half
        np.exp(states, out=states)                                     # <theta|d> = exp[(lam/2)(cos(theta - d) - 1)]
        if case["rule"] in ("born", "peak", "peak_noise", "peak_width", "mean_width"):                                     # amplitudes added, then squared
            percept = np.matmul(amplitude[:, None, :], states)[:, 0, :]
            percept /= np.linalg.norm(percept, axis=1, keepdims=True)
            mem = percept if mem is None else np.sqrt(p["mu"]) * mem + np.sqrt(1 - p["mu"]) * percept
            mem /= np.linalg.norm(mem, axis=1, keepdims=True)
            prob = np.abs(mem) ** 2
        else:                                                          # "decohered": probabilities added
            now = np.matmul((np.abs(amplitude) ** 2)[:, None, :], states ** 2)[:, 0, :]
            now /= now.sum(1, keepdims=True)
            mem = now if mem is None else p["mu"] * mem + (1 - p["mu"]) * now
            prob = mem
        pick = 0
        if case["rule"] == "peak":                                     # always take the most likely heading
            pick = prob.argmax(1)
        elif case["rule"] == "peak_noise":                             # most likely heading + fixed von Mises noise
            pick = None
            theta = rng.vonmises(dirs[prob.argmax(1)], p["lam"]) % (2 * np.pi)
        elif case["rule"] == "peak_width":                             # most likely heading + noise as wide as |Psi|^2
            pick = None
            q = prob / prob.sum(1, keepdims=True)
            Rbar = np.clip(np.hypot(q @ np.cos(dirs), q @ np.sin(dirs)), 1e-6, 1 - 1e-9)
            kap = np.where(Rbar < 0.53, 2 * Rbar + Rbar**3 + 5 * Rbar**5 / 6,
                  np.where(Rbar < 0.85, -0.4 + 1.39 * Rbar + 0.43 / (1 - Rbar), 1 / (Rbar**3 - 4 * Rbar**2 + 3 * Rbar)))
            theta = rng.vonmises(dirs[prob.argmax(1)], kap) % (2 * np.pi)
        elif case["rule"] == "mean_width":                             # expected heading + noise as wide as |Psi|^2
            pick = None
            q = prob / prob.sum(1, keepdims=True)
            C_, S_ = q @ np.cos(dirs), q @ np.sin(dirs)
            Rbar = np.clip(np.hypot(C_, S_), 1e-6, 1 - 1e-9)
            kap = np.where(Rbar < 0.53, 2 * Rbar + Rbar**3 + 5 * Rbar**5 / 6,
                  np.where(Rbar < 0.85, -0.4 + 1.39 * Rbar + 0.43 / (1 - Rbar), 1 / (Rbar**3 - 4 * Rbar**2 + 3 * Rbar)))
            theta = rng.vonmises(np.arctan2(S_, C_), kap) % (2 * np.pi)
        else:
            cdf = np.cumsum(prob, axis=1)
            pick = np.minimum((cdf < rng.random((N, 1)) * cdf[:, -1:]).sum(1), len(dirs) - 1)
        if pick is not None:
            theta = (dirs[pick] + (dirs[1] - dirs[0]) * (rng.random(N) - 0.5)) % (2 * np.pi)

    # 5. Move. Bodies closer than one body length are moved apart until no overlap is left.
    X = X + p["v0"] * np.c_[np.cos(theta), np.sin(theta)]
    for _ in range(p["passes"]):
        pairs = cKDTree(X).query_pairs(p["r0"], output_type="ndarray")
        if not len(pairs):
            break
        i, j = pairs.T
        d = X[j] - X[i]
        L = np.linalg.norm(d, axis=1, keepdims=True) + 1e-12
        push = 0.5 * (p["r0"] - L) * d / L
        np.add.at(X, i, -push)
        np.add.at(X, j, push)
    return X, theta, mem

## 3. Measures and one run

- **Mixing entropy** $M$: for each agent, the Shannon entropy of the kinds among its 10 nearest neighbours, divided by $\ln G$, averaged over agents. 1 = fully mixed, 0 = every neighbourhood holds one kind.
- **Cohesion** $C$: fraction of agents in the largest group, where agents within sight range $R$ of each other are linked.
- **Alignment** $\Phi = |\sum_i \hat e(\theta_i)|/N$.
- **Perception strength** $\mathcal S = |\sum_i \langle P_i|\mathbf v|P_i\rangle|/(N v_0)$: the same quantity computed from the perceptual states, before any heading is drawn.
- **Spacing**: the smallest and the largest distance between the centres of two kinds.
- At the end of a run: the number of **groups by contact** (agents closer than 3 body lengths are linked) and the median **nearest-neighbour distance**.

The final value of a run is the average over its last 20 %.

In [ ]:
def mixing_entropy(X, kind, G, m=10):
    m = min(m, len(X) - 1)
    _, near = cKDTree(X).query(X, m + 1)
    frac = np.stack([(kind[near[:, 1:]] == g).mean(1) for g in range(G)], 1)
    with np.errstate(divide="ignore", invalid="ignore"):
        H = -np.where(frac > 0, frac * np.log(frac), 0).sum(1)
    return H.mean() / np.log(G)

def groups(X, link):
    pairs = cKDTree(X).query_pairs(link, output_type="ndarray")
    A = coo_matrix((np.ones(len(pairs)), (pairs[:, 0], pairs[:, 1])), shape=(len(X),) * 2)
    return connected_components(A, directed=False)[1]

def cohesion(X, link):
    return np.bincount(groups(X, link)).max() / len(X)

def perception_strength(mem, grid):
    prob = np.abs(mem) ** 2
    norm = prob.sum(1)
    return float(np.hypot((prob @ grid[1] / norm).mean(), (prob @ grid[2] / norm).mean()))

def run(case_name, seed, p, T, every=None, frame_every=0, keep_final=False):
    every = EVERY if every is None else every
    case = CASES[case_name]
    Gk = p["G"]
    dirs = np.linspace(0, 2 * np.pi, p["n_dir"], endpoint=False)
    grid = (dirs, np.cos(dirs), np.sin(dirs))
    X, theta, kind, rng = start(p, seed)
    s = similarity_s(Gk, p["delta0"]) if case["similarity"] else 1.0
    S = np.where(kind[:, None] == kind[None, :], 1.0, s)
    mem, frames = None, []
    rec = {"t": [], "M": [], "C": [], "Phi": [], "S": [], "gap_min": [], "gap_max": []}
    for t in range(T + 1):
        if t % every == 0 or t == T:
            rec["t"].append(t)
            rec["M"].append(mixing_entropy(X, kind, Gk))
            rec["C"].append(cohesion(X, p["R"]))
            rec["Phi"].append(np.hypot(np.cos(theta).mean(), np.sin(theta).mean()))
            rec["S"].append(perception_strength(mem, grid) if (mem is not None and case["rule"] == "born") else np.nan)
            cen = np.array([X[kind == g].mean(0) for g in range(Gk)])
            dd = np.linalg.norm(cen[:, None] - cen[None, :], axis=2)[np.triu_indices(Gk, 1)]
            rec["gap_min"].append(dd.min()); rec["gap_max"].append(dd.max())
        if frame_every and t % frame_every == 0:
            frames.append((X.copy(), theta.copy()))
        if t < T:
            X, theta, mem = step(X, theta, mem, kind, S, case, rng, grid, p)
    out = dict(rec={k: np.array(v) for k, v in rec.items()})
    out["M_random"] = float(np.mean([mixing_entropy(X, rng.permutation(kind), Gk) for _ in range(5)]))
    out["contact_groups"] = int(len(np.unique(groups(X, CONTACT * p["r0"]))))
    out["nn"] = float(np.median(cKDTree(X).query(X, 2)[0][:, 1]))
    if keep_final or frame_every:
        out["final"] = (X, theta, kind)
    if frame_every:
        out["frames"] = frames
    return out

def final(r, key, frac=0.2):
    y = r["rec"][key]
    return float(y[-max(1, int(round(len(y) * frac))):].mean())

## 4. Running many simulations in parallel

A job is `(case, seed, parameter changes, T)`; the changes are a tuple of `(name, value)` pairs. Each finished run is saved at once under a key made from all of its parameters, so a saved run is reused only when everything is identical and the notebook can be interrupted at any time.

In [ ]:
def job_params(changes):
    p = dict(BASE); p.update(dict(changes)); return p

def job_key(case_name, seed, changes, T, keep_final):
    payload = dict(p=job_params(changes), case=CASES[case_name], seed=seed, T=T, every=EVERY, final=keep_final, contact=CONTACT)
    return hashlib.md5(json.dumps(payload, sort_keys=True).encode()).hexdigest()

def _work(case_name, case, seed, params, T, every, keep_final, path):
    CASES[case_name] = case                                  # cases defined later in the notebook reach the workers this way
    out = run(case_name, seed, params, T, every=every, keep_final=keep_final)
    with open(path, "wb") as fh:
        pickle.dump(out, fh)
    return out

ALL_RUNS = []                                                # one line per run, saved as all_runs.csv in Section 17

def run_jobs(jobs, label, keep_final=False):
    # jobs: list of (case, seed, changes, T). Returns {job: result}.
    path = lambda j: os.path.join(OUT, "cache", job_key(*j, keep_final) + ".pkl")
    results, todo = {}, []
    for j in jobs:
        if os.path.exists(path(j)):
            with open(path(j), "rb") as fh:
                results[j] = pickle.load(fh)
        else:
            todo.append(j)
    print(f"[{label}] {len(jobs)} runs: {len(jobs) - len(todo)} loaded, {len(todo)} to compute", flush=True)
    todo.sort(key=lambda j: -job_params(j[2])["N"] ** 2 * j[3])           # slowest runs first
    t0 = time.time()
    args = lambda j: (j[0], CASES[j[0]], j[1], job_params(j[2]), j[3], EVERY, keep_final, path(j))
    try:
        from joblib import Parallel, delayed
        cores = os.cpu_count() if N_JOBS == -1 else N_JOBS
        for b in range(0, len(todo), 4 * cores):
            chunk = todo[b:b + 4 * cores]
            outs = Parallel(n_jobs=N_JOBS)(delayed(_work)(*args(j)) for j in chunk)
            results.update(dict(zip(chunk, outs)))
            print(f"   {min(b + 4 * cores, len(todo))}/{len(todo)} done  ({(time.time() - t0) / 60:.1f} min)", flush=True)
    except ImportError:
        print("   joblib not found: running one after another")
        for k, j in enumerate(todo):
            results[j] = _work(*args(j))
            if (k + 1) % 10 == 0 or k + 1 == len(todo):
                print(f"   {k + 1}/{len(todo)} done  ({(time.time() - t0) / 60:.1f} min)", flush=True)
    for j in jobs:
        r, p = results[j], job_params(j[2])
        ALL_RUNS.append({"section": label, "case": j[0], "seed": j[1], "changes": "; ".join(f"{k}={v}" for k, v in j[2]), "steps": j[3],
                         "N": p["N"], "R": p["R"], **{k: final(r, k) for k in ["M", "C", "Phi", "S", "gap_min", "gap_max"]},
                         "groups_by_contact": r["contact_groups"], "nearest_neighbour_distance": r["nn"]})
    return results

def wilson(k, n, z=1.96):
    # 95 % confidence interval of a proportion k/n
    q = k / n; d = 1 + z**2 / n; mid = (q + z**2 / (2 * n)) / d
    h = z * np.sqrt(q * (1 - q) / n + z**2 / (4 * n**2)) / d
    return max(0.0, mid - h), min(1.0, mid + h)

def summary(rs):
    v = {k: np.array([final(r, k) for r in rs]) for k in ["M", "C", "Phi", "S", "gap_min", "gap_max"]}
    v["n"] = len(rs); v["one"] = int((v["C"] > C_ONE).sum())
    v["contact"] = np.array([r["contact_groups"] for r in rs]); v["nn"] = np.array([r["nn"] for r in rs])
    return v

def pm(x):
    return f"{np.mean(x):.2f} ± {np.std(x):.2f}"

def one_group(v):
    lo, hi = wilson(v["one"], v["n"])
    return f"{v['one']}/{v['n']} [{100 * lo:.0f}-{100 * hi:.0f}%]"

def row(label, v):
    return {"case": label, "M": pm(v["M"]), "C": pm(v["C"]), "Phi": pm(v["Phi"]), "one group [95% CI]": one_group(v),
            "groups by contact (median)": int(np.median(v["contact"])), "nearest-neighbour distance": f"{np.median(v['nn']):.2f}"}

def save_fig(fig, name):
    fig.savefig(os.path.join(OUT, name), bbox_inches="tight"); plt.show()

def style(ax):
    ax.yaxis.grid(True, color=GRID, lw=0.6); ax.set_axisbelow(True)

## 5. The model and its controls (Table 2, Figures 2 and 3)

Same seeds, same starting states and same parameters for all cases.

| Case | Partners | Similarity | Identity phase |
|---|---|---|---|
| no identity | sampled from the whole view | ignored | off |
| relational partners, no phase | sampled from the whole view | used | off |
| other kinds' bearings zero weight | sampled from the whole view | used | weight 0 instead of the phase |
| **full model** | sampled from the whole view | used | on |
| nearest partners | 6 nearest | — | on |
| nearest partners, no identity | 6 nearest | — | off |
| half nearest, half relational | 3 nearest + 3 sampled | used | on |

In [ ]:
main_jobs = [(c, s, (), T_RUN) for c in TABLE1 for s in MAIN_SEEDS]
main = run_jobs(main_jobs, "model and controls", keep_final=True)
R = {c: [main[(c, s, (), T_RUN)] for s in MAIN_SEEDS] for c in TABLE1}
SUM = {c: summary(R[c]) for c in TABLE1}
M_RANDOM = float(np.mean([r["M_random"] for r in R["no_identity"]]))

table1 = pd.DataFrame([row(LABEL[c], SUM[c]) for c in TABLE1]); table1.to_csv(os.path.join(OUT, "table1_cases.csv"), index=False)
print(f"Table 2. N = {BASE['N']}, G = {G}, {len(MAIN_SEEDS)} runs per case, {T_RUN} steps. Randomly placed kinds give M = {M_RANDOM:.2f}.")
display(table1)
v = SUM["full"]
spacing_line = (f"Full model, distance between the centres of two kinds: smallest {pm(v['gap_min'])}, largest {pm(v['gap_max'])} body lengths "
                f"(sight range {BASE['R']:g}).")
print(spacing_line)

from scipy.stats import wilcoxon, spearmanr

def mw(a, b):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        U, pv = mannwhitneyu(a, b, alternative="two-sided")
    return pv, 2 * U / (len(a) * len(b)) - 1          # rank-biserial r: -1 = a lower in every pair of runs
def paired(a, b):
    # same starting states, so the runs can also be compared in pairs (Wilcoxon signed-rank test)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return wilcoxon(a, b).pvalue, spearmanr(a, b)[0]
tests = pd.DataFrame([{"mixing entropy": f"{LABEL[a]}  vs  {LABEL[b]}",
                       "Mann-Whitney p": f"{mw(SUM[a]['M'], SUM[b]['M'])[0]:.1e}",
                       "rank-biserial r": f"{mw(SUM[a]['M'], SUM[b]['M'])[1]:+.2f}",
                       "Wilcoxon p (paired)": f"{paired(SUM[a]['M'], SUM[b]['M'])[0]:.1e}",
                       "Spearman (paired runs)": f"{paired(SUM[a]['M'], SUM[b]['M'])[1]:+.2f}"}
                      for a, b in [("relational", "no_identity"), ("ignore_other", "relational"), ("full", "ignore_other"), ("full", "relational"), ("full", "nearest"), ("full", "half_nearest")]])
tests.to_csv(os.path.join(OUT, "table1_tests.csv"), index=False)
display(tests)

### Figure 2 — the flock after 5000 steps

One run (`SHOW_SEED`). Each panel has its own scale. If a flock has split, the largest group is shown and the title says how many agents are in view.

In [ ]:
def view(X, R_sight, pad=4.0, min_half=15.0, max_half=80.0):
    # centre and half-width of a window that shows all agents if they fit, otherwise the largest group
    ctr = X.mean(0); half = np.abs(X - ctr).max() + pad
    if half <= max_half:
        return ctr, max(half, min_half), 1.0
    lab = groups(X, R_sight); big = lab == np.bincount(lab).argmax()
    ctr = X[big].mean(0); half = np.abs(X[big] - ctr).max() + pad
    return ctr, min(max(half, min_half), max_half), big.mean()

def draw_flock(ax, X, theta, kind, title):
    ax.quiver(X[:, 0], X[:, 1], np.cos(theta), np.sin(theta), color=[KIND_COLORS[k % len(KIND_COLORS)] for k in kind],
              angles="xy", scale_units="xy", scale=0.5, width=0.005)
    ctr, h, shown = view(X, BASE["R"])
    ax.set_xlim(ctr[0] - h, ctr[0] + h); ax.set_ylim(ctr[1] - h, ctr[1] + h)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_color("#c3c2b7")
    ax.plot([ctr[0] - 0.9 * h, ctr[0] - 0.9 * h + 10], [ctr[1] - 0.9 * h] * 2, color=INK, lw=2)
    ax.text(ctr[0] - 0.9 * h, ctr[1] - 0.86 * h, "10 body lengths", ha="left", va="bottom", fontsize=8, color=INK2)
    ax.set_title(title + ("" if shown > 0.995 else f"\n({100 * shown:.0f}% of agents in view)"), loc="left", fontsize=9, color=INK)

show = MAIN_SEEDS.index(SHOW_SEED) if SHOW_SEED in MAIN_SEEDS else 0
fig, ax = plt.subplots(1, len(SHOW), figsize=(3.2 * len(SHOW), 3.7))
for a_, c, tag in zip(ax, SHOW, "abcd"):
    r = R[c][show]; X, th, kd = r["final"]
    draw_flock(a_, X, th, kd, f"({tag}) {LABEL[c]}\n$M$ = {final(r, 'M'):.2f},  $C$ = {final(r, 'C'):.2f},  $\\Phi$ = {final(r, 'Phi'):.2f}")
fig.tight_layout(w_pad=0.8); save_fig(fig, "fig2_flocks.png")

### Figure 3 — mixing, cohesion and alignment over time

Mean over all runs (line) and ± one standard deviation (band).

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(9.6, 3.0))
for a_, (key, name) in zip(ax, [("M", "mixing entropy $M$"), ("C", "cohesion $C$"), ("Phi", r"alignment $\Phi$")]):
    for c in SHOW:
        Y = np.array([r["rec"][key] for r in R[c]]); t = R[c][0]["rec"]["t"]
        a_.fill_between(t, Y.mean(0) - Y.std(0), Y.mean(0) + Y.std(0), color=COLOR[c], alpha=0.15, lw=0)
        a_.plot(t, Y.mean(0), color=COLOR[c], lw=2.4 if c == "full" else 1.8, ls=DASH[c], label=LABEL[c])
    a_.set_ylim(0, 1.05); a_.set_xlabel("time (steps)"); a_.set_ylabel(name); style(a_)
ax[0].axhline(M_RANDOM, color=INK2, lw=0.8, ls=(0, (1, 2)))
ax[0].legend(loc="center right", fontsize=7.5, handlelength=1.8)
fig.tight_layout(); save_fig(fig, "fig3_over_time.png")
t_sorted = [int(r["rec"]["t"][np.argmax(r["rec"]["M"] < 0.05)]) if (r["rec"]["M"] < 0.05).any() else None for r in R["full"]]
done = [x for x in t_sorted if x is not None]
sorting_time_line = (f"Full model: mixing entropy falls below 0.05 in {len(done)}/{len(t_sorted)} runs; "
                     f"time needed: median {np.median(done):.0f} steps, largest {np.max(done):.0f} steps." if done else "Full model: M never falls below 0.05.")
print(sorting_time_line)

## 6. Decision rules without interference (Table 3)

Same partners, same perceived states and same parameters as the full model; only the decision rule changes (see Section 2). The last four rows keep the model's perceptual state and memory but replace the draw: always the most likely heading (`peak`), that heading plus noise of fixed width (`peak_noise`), that heading plus noise as wide as the draw (`peak_width`), and the expected heading plus noise as wide as the draw (`mean_width`). The three `_noid` runs repeat `peak`, `peak_noise` and `mean_width` without identity; they are quoted in the text (Section 5.5).

Below the table: how many runs segregate ($M<0.05$), with Fisher's exact test and the Mann–Whitney test against the full model.


In [ ]:
RULE_ROWS = ["decohered", "vector", "vector_zero", "peak", "peak_noise", "peak_width", "mean_width"]
EXTRA = ["peak_noid", "peak_noise_noid", "mean_width_noid"]
rule_jobs = [(c, s, (), T_RUN) for c in RULE_ROWS + EXTRA for s in MAIN_SEEDS]
rules = run_jobs(rule_jobs, "decision rules")
RULES = {"full": SUM["full"]}
RULES.update({c: summary([rules[(c, s, (), T_RUN)] for s in MAIN_SEEDS]) for c in RULE_ROWS + EXTRA})
names = {"full": "amplitudes added, then squared (the model)", **{c: LABEL[c] for c in RULE_ROWS + EXTRA}}
table2 = pd.DataFrame([row(names[c], RULES[c]) for c in ["full"] + RULE_ROWS])
table2 = table2.rename(columns={"case": "decision rule"}); table2.to_csv(os.path.join(OUT, "table2_rules.csv"), index=False)
print(f"Table 3. Decision rules, {len(MAIN_SEEDS)} runs each (same seeds as Table 2).")
display(table2)

from scipy.stats import fisher_exact
Mfull = RULES["full"]["M"]; seg = lambda m: int((np.asarray(m) < 0.05).sum()); nrun = len(MAIN_SEEDS)
tests2 = pd.DataFrame([{"rule": names[c], "runs with M < 0.05": f"{seg(RULES[c]['M'])}/{nrun}",
                        "Fisher p (vs model)": f"{fisher_exact([[seg(Mfull), nrun - seg(Mfull)], [seg(RULES[c]['M']), nrun - seg(RULES[c]['M'])]])[1]:.1g}",
                        "Mann-Whitney p on M (vs model)": f"{mannwhitneyu(Mfull, RULES[c]['M']).pvalue:.1g}"} for c in RULE_ROWS])
tests2.to_csv(os.path.join(OUT, "table2_rules_tests.csv"), index=False)
display(tests2)
for c in EXTRA:
    print(f"{names[c]}: M = {pm(RULES[c]['M'])}, one group {one_group(RULES[c])}")


## 7. How far must an agent see? (Figure 4)

The sight range $R$ is varied for three flock sizes. The values of $R$ are set as the width of the starting disc divided by a ratio, so each flock size is tested from 1.5 to 5 sight ranges wide. The figure is drawn against $R$ itself: the results for the three flock sizes fall on one curve against $R$, and not against the ratio.

In [ ]:
limit_jobs, limit_index = [], {}
for N_, (ratios, n_runs) in LIMIT_PLAN.items():
    for ratio in ratios:
        R_ = round(float(start_width(N_) / ratio), 3)
        for c in ["no_identity", "full"]:
            js = [(c, LIMIT_SEED0 + k, (("N", N_), ("R", R_)), T_RUN) for k in range(n_runs)]
            limit_jobs += js; limit_index[(N_, ratio, c)] = js
limit = run_jobs(limit_jobs, "flock width / sight range")
LIM = {key: summary([limit[j] for j in js]) for key, js in limit_index.items()}

rows = []
for (N_, ratio, c), v in LIM.items():
    q = np.percentile(v["M"], [25, 50, 75])
    rows.append({"N": N_, "width / sight range": ratio, "sight range R": f"{start_width(N_) / ratio:.1f}", "case": LABEL[c],
                 "M median [quartiles]": f"{q[1]:.2f} [{q[0]:.2f}-{q[2]:.2f}]", "C": pm(v["C"]), "Phi": pm(v["Phi"]), "one group [95% CI]": one_group(v)})
table_limit = pd.DataFrame(rows).sort_values(["case", "N", "width / sight range"]); table_limit.to_csv(os.path.join(OUT, "table_size_limit.csv"), index=False)
display(table_limit)

sizes = list(LIMIT_PLAN)
mk = dict(zip(sizes, ["o", "s", "^", "D"])); shade = dict(zip(sizes, ["#9085e9", "#4a3aa7", "#1f1750", "#000000"]))
grey = dict(zip(sizes, ["#b5b4ae", "#7a7974", "#3d3d3a", "#000000"]))
fig, ax = plt.subplots(1, 3, figsize=(10.4, 3.2))
for N_ in sizes:
    ratios = LIMIT_PLAN[N_][0]; xs = [start_width(N_) / x for x in ratios]
    for a_, c, col in [(ax[0], "full", shade[N_]), (ax[1], "no_identity", grey[N_])]:
        pct = np.array([100 * LIM[(N_, x, c)]["one"] / LIM[(N_, x, c)]["n"] for x in ratios])
        ci = np.array([wilson(LIM[(N_, x, c)]["one"], LIM[(N_, x, c)]["n"]) for x in ratios]) * 100
        a_.errorbar(xs, pct, yerr=[np.clip(pct - ci[:, 0], 0, None), np.clip(ci[:, 1] - pct, 0, None)], color=col, lw=1.8, marker=mk[N_], ms=5,
                    mec="white", mew=0.6, capsize=2.5, elinewidth=0.8, label=f"$N$ = {N_}")
    q = np.array([np.percentile(LIM[(N_, x, "full")]["M"], [25, 50, 75]) for x in ratios])
    ax[2].errorbar(xs, q[:, 1], yerr=[q[:, 1] - q[:, 0], q[:, 2] - q[:, 1]], color=shade[N_], lw=1.8, marker=mk[N_], ms=5, mec="white", mew=0.6,
                   capsize=2.5, elinewidth=0.8, label=f"$N$ = {N_}")
ax[0].set_ylabel("runs that ended as one group (%)"); ax[0].set_title("(a) full model", loc="left", fontsize=9)
ax[1].set_ylabel("runs that ended as one group (%)"); ax[1].set_title("(b) no identity", loc="left", fontsize=9)
ax[2].set_ylabel("mixing entropy $M$ (median, quartiles)"); ax[2].set_title("(c) full model", loc="left", fontsize=9)
ax[2].axhline(M_RANDOM, color=INK2, lw=0.8, ls=(0, (1, 2)))
for a_ in ax:
    a_.set_xlabel("sight range $R$ (body lengths)"); style(a_)
ax[0].set_ylim(-3, 105); ax[1].set_ylim(-3, 105); ax[2].set_ylim(0, 1.05)
ax[0].legend(fontsize=8, loc="lower right"); ax[1].legend(fontsize=8, loc="lower right"); ax[2].legend(fontsize=8, loc="center right")
fig.tight_layout(w_pad=1.5); save_fig(fig, "fig4_sight_range.png")

# the full model pooled over flock sizes, in three bands of sight range
def band(lo, hi):
    keys = [(N_, x) for N_ in sizes for x in LIMIT_PLAN[N_][0] if lo <= start_width(N_) / x < hi]
    return sum(LIM[(N_, x, "full")]["one"] for N_, x in keys), sum(LIM[(N_, x, "full")]["n"] for N_, x in keys)
sight_line = "Full model, runs that ended as one group, all flock sizes together:  " + ";  ".join(
    f"R {name}: {band(lo, hi)[0]}/{band(lo, hi)[1]}" for name, lo, hi in [("below 11", 0, 11), ("11 to 15", 11, 15), ("15 and above", 15, 1e9)])
print(sight_line)

## 8. How much attention to other kinds? (Figure 5)

The full model for different $\delta_0$. The two ends are limits of the model:
- $\delta_0 = 0$: partners of the agent's own kind only.
- **random**: similarity ignored ($s = 1$, that is $\delta_0 = (G-1)/G$), partners drawn uniformly from the view, identity phase kept.

In [ ]:
def delta_value(d):
    return (G - 1) / G if d == "random" else d

sweep_jobs = [("full", s, (("delta0", delta_value(d)),), T_RUN) for d in SWEEP_DELTAS for s in SWEEP_SEEDS]
sweep = run_jobs(sweep_jobs, "delta0 sweep")
SW = {d: summary([sweep[("full", s, (("delta0", delta_value(d)),), T_RUN)] for s in SWEEP_SEEDS]) for d in SWEEP_DELTAS}

rows = []
for d in SWEEP_DELTAS:
    v = SW[d]; q = np.percentile(v["M"], [25, 50, 75])
    rows.append({"delta0": str(d), "s": f"{similarity_s(G, delta_value(d)):.3f}", "M median [quartiles]": f"{q[1]:.2f} [{q[0]:.2f}-{q[2]:.2f}]",
                 "C median": f"{np.median(v['C']):.2f}", "Phi": pm(v["Phi"]), "one group [95% CI]": one_group(v)})
table_attention = pd.DataFrame(rows); table_attention.to_csv(os.path.join(OUT, "table_attention.csv"), index=False)
print(f"Attention to other kinds (full model, {len(SWEEP_SEEDS)} runs per value).")
display(table_attention)

def two_panels(ax, xs, stats, color="#4a3aa7"):
    q = np.array([np.percentile(v["M"], [25, 50, 75]) for v in stats])
    ax[0].errorbar(xs, q[:, 1], yerr=[q[:, 1] - q[:, 0], q[:, 2] - q[:, 1]], color=color, lw=2, marker="o", ms=5, capsize=3, elinewidth=1)
    pct = np.array([100 * v["one"] / v["n"] for v in stats]); ci = np.array([wilson(v["one"], v["n"]) for v in stats]) * 100
    ax[1].errorbar(xs, pct, yerr=[pct - ci[:, 0], ci[:, 1] - pct], color=color, lw=2, marker="o", ms=5, capsize=3, elinewidth=1)
    ax[0].set_ylim(0, 1.05); ax[1].set_ylim(-3, 105)
    for a_ in ax:
        style(a_)

xs = np.arange(len(SWEEP_DELTAS))
fig, ax = plt.subplots(1, 2, figsize=(8.2, 3.1))
two_panels(ax, xs, [SW[d] for d in SWEEP_DELTAS])
ax[0].axhline(M_RANDOM, color=INK2, lw=0.8, ls=(0, (1, 2))); ax[0].text(len(xs) - 0.6, M_RANDOM + 0.02, "randomly placed kinds", fontsize=8, color=INK2, ha="right")
ax[0].set_ylabel("mixing entropy $M$ (median, quartiles)"); ax[1].set_ylabel("runs that ended as one group (%)")
for a_ in ax:
    a_.set_xticks(xs); a_.set_xticklabels([str(d) for d in SWEEP_DELTAS]); a_.set_xlim(-0.4, len(xs) - 0.6)
    a_.set_xlabel(r"share of attention to other kinds, $\delta_0$")
fig.tight_layout(w_pad=2.0); save_fig(fig, "fig5_attention.png")

## 9. Noise (Figure 6)

The model has no added noise. The heading is drawn from the perceptual state, so the noise is set by the sharpness of perception $\lambda$: when all perceived directions agree, the drawn heading has variance $1/\lambda$, which is the variance of a Vicsek noise of width $\eta = \sqrt{12/\lambda}$. When the perceived directions disagree, the variance is larger: to leading order $\sigma^2 = 1/\lambda + \tfrac12\mathrm{Var}_a(d)$. The first cell checks these two statements; the second varies $\lambda$ in the simulations.

In [ ]:
def heading_variance(a, d, lam):
    # variance of the heading drawn from the percept with amplitudes a and directions d (radians)
    th = np.linspace(-np.pi, np.pi, 100001)
    psi = (a[:, None] * np.exp(0.5 * lam * (np.cos(th[None, :] - d[:, None]) - 1))).sum(0)
    P = psi ** 2; P /= P.sum()
    m = np.arctan2(P @ np.sin(th), P @ np.cos(th)); x = (th - m + np.pi) % (2 * np.pi) - np.pi
    return float(P @ x ** 2)

rng_ = np.random.default_rng(0); lam = BASE["lam"]
a = np.r_[1.0, np.full(6, np.sqrt(BASE["w"]))]; a /= np.linalg.norm(a)
rows = [{"spread of partner headings (deg)": 0, "variance of the drawn heading": f"{heading_variance(a, np.zeros(7), lam):.4f}",
         "1/lambda + Var_a(d)/2": f"{1 / lam:.4f}", "equivalent Vicsek noise eta (deg)": f"{np.degrees(np.sqrt(12 * heading_variance(a, np.zeros(7), lam))):.0f}"}]
for spread in [5, 10, 20]:
    sim, th_ = [], []
    for _ in range(300):
        d = np.r_[0.0, rng_.normal(0, np.radians(spread), 6)]; q = a / a.sum()
        sim.append(heading_variance(a, d, lam)); th_.append(1 / lam + 0.5 * (q @ d ** 2 - (q @ d) ** 2))
    rows.append({"spread of partner headings (deg)": spread, "variance of the drawn heading": f"{np.mean(sim):.4f}",
                 "1/lambda + Var_a(d)/2": f"{np.mean(th_):.4f}", "equivalent Vicsek noise eta (deg)": f"{np.degrees(np.sqrt(12 * np.mean(sim))):.0f}"})
table_noise_map = pd.DataFrame(rows); table_noise_map.to_csv(os.path.join(OUT, "table_noise_map.csv"), index=False)
print(f"Noise of one decision, lambda = {lam:g} (own heading and six partner headings).")
display(table_noise_map)

In [ ]:
noise_jobs = [(c, s, (("lam", l),), T_RUN) for l in NOISE_LAMS for c in ["no_identity", "full"] for s in SENS_SEEDS]
noise = run_jobs(noise_jobs, "noise")
NZ = {(c, l): summary([noise[(c, s, (("lam", l),), T_RUN)] for s in SENS_SEEDS]) for l in NOISE_LAMS for c in ["no_identity", "full"]}
eta = lambda l: float(np.degrees(np.sqrt(12 / l)))
rows = []
for l in NOISE_LAMS:
    for c in ["no_identity", "full"]:
        v = NZ[(c, l)]
        rows.append({"lambda": l, "equivalent eta (deg)": f"{eta(l):.0f}", "case": LABEL[c], "M": pm(v["M"]), "C": pm(v["C"]), "Phi": pm(v["Phi"]), "one group [95% CI]": one_group(v)})
table_noise = pd.DataFrame(rows); table_noise.to_csv(os.path.join(OUT, "table_noise.csv"), index=False)
display(table_noise)

xs = [eta(l) for l in NOISE_LAMS]
fig, ax = plt.subplots(1, 3, figsize=(10.4, 3.1))
for c in ["no_identity", "full"]:
    F = np.array([[NZ[(c, l)]["Phi"].mean(), NZ[(c, l)]["Phi"].std()] for l in NOISE_LAMS])
    ax[0].errorbar(xs, F[:, 0], yerr=F[:, 1], color=COLOR[c], ls=DASH[c], lw=2, marker="o", ms=5, capsize=3, elinewidth=1, label=LABEL[c])
    pct = np.array([100 * NZ[(c, l)]["one"] / NZ[(c, l)]["n"] for l in NOISE_LAMS]); ci = np.array([wilson(NZ[(c, l)]["one"], NZ[(c, l)]["n"]) for l in NOISE_LAMS]) * 100
    ax[2].errorbar(xs, pct, yerr=[pct - ci[:, 0], ci[:, 1] - pct], color=COLOR[c], ls=DASH[c], lw=2, marker="o", ms=5, capsize=3, elinewidth=1, label=LABEL[c])
q = np.array([np.percentile(NZ[("full", l)]["M"], [25, 50, 75]) for l in NOISE_LAMS])
ax[1].errorbar(xs, q[:, 1], yerr=[q[:, 1] - q[:, 0], q[:, 2] - q[:, 1]], color=COLOR["full"], lw=2, marker="o", ms=5, capsize=3, elinewidth=1)
ax[0].set_ylabel(r"alignment $\Phi$"); ax[0].set_ylim(0, 1.05); ax[0].legend(fontsize=8, loc="lower left")
ax[1].set_ylabel("mixing entropy $M$, full model"); ax[1].set_ylim(0, 1.05)
ax[2].set_ylabel("runs that ended as one group (%)"); ax[2].set_ylim(-3, 105)
for a_ in ax:
    a_.set_xlabel(r"noise: equivalent Vicsek $\eta=\sqrt{12/\lambda}$ (degrees)"); style(a_)
fig.tight_layout(w_pad=1.5); save_fig(fig, "fig6_noise.png")

## 10. Where the result holds (Figure 7)

The full model, with one quantity changed at a time:
- **phase** of the bearing of a partner of another kind, from 0 (no identity phase) to $\pi$ (the model);
- **number of partners** $n$;
- **heading weight** $w$;
- **memory** $\mu$;
- **number of kinds** $G$ (equal numbers, 240 agents in all).

In [ ]:
def sens_job(name, value, seed):
    if name == "phase":
        cname = f"phase_{value:.4f}"
        CASES[cname] = dict(CASES["full"], phase=float(value))
        return (cname, seed, (), T_RUN)
    return ("full", seed, ((name, value),), T_RUN)

sens_jobs = [sens_job(k, v, s) for k, vals in SENS.items() for v in vals for s in SENS_SEEDS]
sens = run_jobs(sens_jobs, "sensitivity")
SN = {(k, v): summary([sens[sens_job(k, v, s)] for s in SENS_SEEDS]) for k, vals in SENS.items() for v in vals}
xlab = {"phase": r"identity phase $\varphi$ (units of $\pi$)", "n": "number of partners $n$", "w": "heading weight $w$",
        "mu": r"memory $\mu$", "G": "number of kinds $G$"}
shown = lambda k, v: f"{v / PI:.2f}" if k == "phase" else f"{v:g}"
rows = []
for (k, v), st in SN.items():
    q = np.percentile(st["M"], [25, 50, 75])
    rows.append({"quantity": k, "value": shown(k, v) + (" pi" if k == "phase" else ""), "M median [quartiles]": f"{q[1]:.2f} [{q[0]:.2f}-{q[2]:.2f}]",
                 "C": pm(st["C"]), "Phi": pm(st["Phi"]), "one group [95% CI]": one_group(st)})
table_sens = pd.DataFrame(rows); table_sens.to_csv(os.path.join(OUT, "table_sensitivity.csv"), index=False)
print(f"Full model with one quantity changed ({len(SENS_SEEDS)} runs per value).")
display(table_sens)

keys = list(SENS)
fig, ax = plt.subplots(2, len(keys), figsize=(2.3 * len(keys), 4.6), sharey="row", squeeze=False)
for c_, k in enumerate(keys):
    xs = np.arange(len(SENS[k]))
    two_panels(ax[:, c_], xs, [SN[(k, v)] for v in SENS[k]])
    for a_ in ax[:, c_]:
        a_.set_xticks(xs); a_.set_xticklabels([shown(k, v) for v in SENS[k]], fontsize=7.5); a_.set_xlim(-0.5, len(xs) - 0.5)
    ax[1, c_].set_xlabel(xlab[k])
ax[0, 0].set_ylabel("mixing entropy $M$\n(median, quartiles)"); ax[1, 0].set_ylabel("runs that ended\nas one group (%)")
fig.tight_layout(w_pad=0.8, h_pad=0.8); save_fig(fig, "fig7_sensitivity.png")

## 11. Which cue does what, and a large flock followed for longer

**Cue controls.** The full model with one cue changed (the 30 seeds of Tables 1 and 3; the final positions are kept, so the width of the flock can be measured):
- *no bearings*: $w = 1$, so only headings are perceived;
- *bearings of other kinds ignored*: weight 0 instead of a negative amplitude (no phase);
- *phase on the headings* of partners of other kinds, instead of their bearings, and on both;
- *phase* $\pi/4$, $\pi/2$ and $3\pi/4$ on the bearings of other kinds (0 is the relational case of Table 1, $\pi$ the full model).

**Large flock.** The flock of 960 agents is followed for 15,000 steps at two sight ranges, to see whether the sorting continues after 5,000 steps. At the end of each run the cell reports the share of the flock that an agent sees and the width of the flock.

In [ ]:
CASES["heading_phase"] = dict(CASES["full"], phase_on="headings")
CASES["both_phase"]    = dict(CASES["full"], phase_on="both")
CUES = [("full model", "full", ()),
        ("no bearings (w = 1)", "full", (("w", 1.0),)),
        ("bearings of other kinds ignored (weight 0)", "ignore_other", ()),
        ("phase on the headings of other kinds instead", "heading_phase", ()),
        ("phase on headings and bearings", "both_phase", ())]
for k in (1, 2, 3):
    CASES[f"phase_{k}"] = dict(CASES["full"], phase=k * PI / 4)
    CUES.append((f"phase {k}pi/4 on the bearings of other kinds", f"phase_{k}", ()))
cue_jobs = [(c, s, ch, T_RUN) for _, c, ch in CUES for s in MAIN_SEEDS]
cue = run_jobs(cue_jobs, "cue controls", keep_final=True)

def width_of_largest(r, R_sight):
    # width of the largest group at the end of a run: twice the largest distance from its centre
    X = r["final"][0]; lab = groups(X, R_sight); big = lab == np.bincount(lab).argmax()
    return 2 * np.linalg.norm(X[big] - X[big].mean(0), axis=1).max()

rows = []
for name, c, ch in CUES:
    rs = [cue[(c, s, ch, T_RUN)] for s in MAIN_SEEDS]; v = summary(rs)
    rows.append({"case": name, "M": pm(v["M"]), "C": pm(v["C"]), "Phi": pm(v["Phi"]), "one group [95% CI]": one_group(v),
                 "width of the largest group": pm([width_of_largest(r, BASE["R"]) for r in rs])})
table_cues = pd.DataFrame(rows); table_cues.to_csv(os.path.join(OUT, "table_cues.csv"), index=False)
print(f"Cue controls ({len(MAIN_SEEDS)} runs per row). Width of the starting flock: {start_width(BASE['N']):.0f} body lengths.")
display(table_cues)

# The zero-weight control again, with the 30 starting states of Tables 2 and 3, compared with the full model
zero = run_jobs([("ignore_other", s, (), T_RUN) for s in MAIN_SEEDS], "zero weight, 30 runs")
Mz = np.array([final(zero[("ignore_other", s, (), T_RUN)], "M") for s in MAIN_SEEDS])
Mf = np.array([final(main[("full", s, (), T_RUN)], "M") for s in MAIN_SEEDS])
vz = summary([zero[("ignore_other", s, (), T_RUN)] for s in MAIN_SEEDS])
print(f"zero weight, 30 runs: M = {pm(Mz)} (median {np.median(Mz):.2f}, {(Mz < 0.05).sum()}/30 below 0.05), one group {one_group(vz)}")
print(f"full model, same starts: {(Mf < 0.05).sum()}/30 below 0.05; Mann-Whitney p = {mannwhitneyu(Mf, Mz).pvalue:.1g}")


In [ ]:
def share_in_sight(r, p):
    # mean share of the other agents that an agent sees at the end of a run (sight range and field of view)
    X, th, _ = r["final"]
    dx = X[None, :, 0] - X[:, None, 0]; dy = X[None, :, 1] - X[:, None, 1]
    dist = np.sqrt(dx * dx + dy * dy); np.fill_diagonal(dist, np.inf)
    visible = (dist <= p["R"]) & (dx * np.cos(th)[:, None] + dy * np.sin(th)[:, None] >= np.cos(np.deg2rad(p["fov_deg"]) / 2) * dist)
    return visible.sum() / (len(X) * (len(X) - 1))

big_jobs = {q: [("full", s, (("N", BIG_N), ("R", round(float(start_width(BIG_N) / q), 3))), BIG_T) for s in BIG_SEEDS] for q in BIG_RATIOS}
big = run_jobs([j for js in big_jobs.values() for j in js], "large flock, long runs", keep_final=True)
marks = sorted({m for m in [T_RUN, 2 * BIG_T // 3, BIG_T] if m <= BIG_T})
rows = []
for q, js in big_jobs.items():
    p = job_params(js[0][2]); rs = [big[j] for j in js]
    at = lambda r, key, m: r["rec"][key][min(np.searchsorted(r["rec"]["t"], m), len(r["rec"]["t"]) - 1)]
    row = {"N": BIG_N, "sight range R": f"{p['R']:.1f}", "starting width / R": q}
    for m in marks:
        row[f"M at step {m}"] = pm([at(r, "M", m) for r in rs])
    W = [width_of_largest(r, p["R"]) for r in rs]
    row.update({"C at the end": pm([at(r, "C", BIG_T) for r in rs]), "one group": f"{sum(at(r, 'C', BIG_T) > C_ONE for r in rs)}/{len(rs)}",
                "share of the flock an agent sees": pm([share_in_sight(r, p) for r in rs]),
                "width of the flock": pm(W), "width / R": pm(np.array(W) / p["R"])})
    rows.append(row)
table_big = pd.DataFrame(rows); table_big.to_csv(os.path.join(OUT, "table_large_flock.csv"), index=False)
print(f"Large flock followed for {BIG_T} steps (full model, {len(BIG_SEEDS)} runs per row); values at single steps.")
display(table_big)

## 12. Perception strength and alignment (Figure 8)

A consistency check. The alignment $\Phi$ is measured from the headings the agents take. The perception strength $\mathcal S$ is computed from their perceptual states, before any heading is drawn. Because the headings are drawn from the perceptual states, the two must agree in a large flock up to sampling fluctuations.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8.4, 3.5), gridspec_kw={"width_ratios": [1.35, 1]})
rows = []
for c in SHOW:
    t = R[c][0]["rec"]["t"][1:]
    S_ = np.array([r["rec"]["S"][1:] for r in R[c]]); F_ = np.array([r["rec"]["Phi"][1:] for r in R[c]])
    ax[0].plot(t, S_.mean(0), color=COLOR[c], lw=2)
    k = np.unique(np.geomspace(1, len(t), 14).astype(int) - 1)             # markers evenly spaced on the logarithmic time axis
    ax[0].plot(t[k], F_.mean(0)[k], ls="none", marker="o", ms=5, mfc="white", mec=COLOR[c], mew=1.3)
    ax[1].scatter(F_.ravel(), S_.ravel(), s=5, color=COLOR[c], alpha=0.25, lw=0, rasterized=True)
    rows.append({"case": LABEL[c], "perception strength S": pm(SUM[c]["S"]), "alignment Phi": pm(SUM[c]["Phi"]),
                 "mean |S - Phi| over all times": f"{float(np.abs(S_ - F_).mean()):.3f}",
                 "correlation of S and Phi over all times": f"{float(np.corrcoef(F_.ravel(), S_.ravel())[0, 1]):.3f}"})
ax[0].set_xscale("log"); ax[0].set_ylim(0, 1.05); ax[0].set_xlabel("time (steps, logarithmic)"); ax[0].set_ylabel("order of the flock"); style(ax[0])
handles = [plt.Line2D([], [], color=COLOR[c], lw=2, ls="-", label=LABEL[c]) for c in SHOW]
handles += [plt.Line2D([], [], color=INK2, lw=2, label=r"perception strength $\mathcal{S}$"),
            plt.Line2D([], [], ls="none", marker="o", ms=5, mfc="white", mec=INK2, mew=1.3, label=r"alignment $\Phi$")]
ax[0].legend(handles=handles, loc="lower right", fontsize=7.5, handlelength=1.8)
ax[1].plot([0, 1], [0, 1], color=INK, lw=0.8, ls=(0, (4, 2)), zorder=0)
ax[1].set_xlim(0, 1); ax[1].set_ylim(0, 1); ax[1].set_aspect("equal")
ax[1].set_xlabel(r"alignment $\Phi$ (physical space)"); ax[1].set_ylabel(r"perception strength $\mathcal{S}$ (perceptual space)")
ax[1].text(0.52, 0.44, r"$\mathcal{S} = \Phi$", fontsize=8, color=INK, rotation=45, ha="center", va="center")
fig.tight_layout(w_pad=1.5); save_fig(fig, "fig8_perception_strength.png")
table_strength = pd.DataFrame(rows); table_strength.to_csv(os.path.join(OUT, "table_perception_strength.csv"), index=False)
display(table_strength)

## 13. Long runs (Figure 9)

The full model continued to 15,000 steps. Besides the three measures, the distance between the centres of the kinds is followed: if the sorted groups drifted apart, it would grow.

In [ ]:
long_jobs = [("full", s, (), LONG_T) for s in LONG_SEEDS]
long_runs = run_jobs(long_jobs, "long runs")
LR = [long_runs[j] for j in long_jobs]
marks = sorted({m for m in [T_RUN, 2 * LONG_T // 3, LONG_T] if m <= LONG_T})
rows = []
for m in marks:
    vals = {k: [] for k in ["M", "C", "Phi", "gap_min", "gap_max"]}
    for r in LR:
        q = min(np.searchsorted(r["rec"]["t"], m), len(r["rec"]["t"]) - 1)
        for k in vals:
            vals[k].append(r["rec"][k][q])
    together = np.array(vals["C"]) > C_ONE
    rows.append({"step": m, "M": pm(vals["M"]), "C": pm(vals["C"]), "Phi": pm(vals["Phi"]), "one group": f"{int(together.sum())}/{len(LR)}",
                 "smallest distance between kinds (runs still one group)": pm(np.array(vals["gap_min"])[together]) if together.any() else "-",
                 "largest distance between kinds (runs still one group)": pm(np.array(vals["gap_max"])[together]) if together.any() else "-"})
table_long = pd.DataFrame(rows); table_long.to_csv(os.path.join(OUT, "table_long_runs.csv"), index=False)
print(f"Long runs (full model, {len(LR)} runs), values at single steps.")
display(table_long)

# trend of the largest distance between kinds after the sorting, in the runs that stay one group to the end
late = lambda r: r["rec"]["t"] >= T_RUN
trend = [1000 * np.polyfit(r["rec"]["t"][late(r)], r["rec"]["gap_max"][late(r)], 1)[0] for r in LR if r["rec"]["C"][-1] > C_ONE and late(r).sum() > 2]
trend_line = (f"Trend of the largest distance between kinds after step {T_RUN}, runs that stay one group: "
              f"{np.mean(trend):+.2f} ± {np.std(trend):.2f} body lengths per 1000 steps ({len(trend)} runs)." if trend else "No run stayed one group.")
print(trend_line)

fig, ax = plt.subplots(1, 2, figsize=(8.4, 3.0))
for r in LR:
    lost = r["rec"]["C"][-1] <= C_ONE
    ax[0].plot(r["rec"]["t"], r["rec"]["gap_max"], color="#eb6834" if lost else "#4a3aa7", lw=0.9, alpha=0.8)
    ax[1].plot(r["rec"]["t"], r["rec"]["C"], color="#eb6834" if lost else "#4a3aa7", lw=0.9, alpha=0.8)
ax[0].axhline(BASE["R"], color=INK2, lw=0.8, ls=(0, (1, 2))); ax[0].text(LONG_T, BASE["R"] * 1.03, "sight range", fontsize=8, color=INK2, ha="right")
ax[0].set_ylim(0, 3 * BASE["R"]); ax[0].set_ylabel("largest distance between kinds"); ax[1].set_ylabel("cohesion $C$"); ax[1].set_ylim(0, 1.05)
for a_ in ax:
    a_.set_xlabel("time (steps)"); style(a_)
fig.tight_layout(w_pad=1.5); save_fig(fig, "fig9_long_runs.png")

## 14. One decision worked through, and Figure 1

An agent of kind A at the origin, heading 0°, with six partners (three of kind A, three of kind B). The cell

- lists the perceived states and their amplitudes,
- gives the heading probabilities with and without the identity phase,
- checks the operator form of the expected velocity,
$$\langle \mathbf v_i\rangle = v_0\,\frac{\mathrm{Tr}(\rho_i\,\mathbf O_i)}{\mathrm{Tr}(\rho_i W_i)},\qquad
(\mathbf O_i)_{mn} = I_1\!\big(\lambda\cos\tfrac{\Delta_{mn}}{2}\big)\,\hat e(\bar d_{mn}),\qquad (W_i)_{mn} = I_0\!\big(\lambda\cos\tfrac{\Delta_{mn}}{2}\big),$$
against the direct sum over headings ($\rho_{mn} = a_m a_n$; $\Delta_{mn}$: angle between perceived directions $m$ and $n$; $\bar d_{mn}$: their bisector),
- and draws Figure 1 from the same numbers.

In [ ]:
def operators(d, lam):
    # perception operator (Ox, Oy) and overlap matrix W for perceived directions d (radians)
    d = np.asarray(d)
    delta = (d[:, None] - d[None, :] + np.pi) % (2 * np.pi) - np.pi
    x = lam * np.cos(delta / 2)
    ex = np.cos(d)[:, None] + np.cos(d)[None, :]; ey = np.sin(d)[:, None] + np.sin(d)[None, :]
    nrm = np.maximum(np.hypot(ex, ey), 1e-300)
    scale = np.exp(x - lam)                                   # i0e, i1e are exponentially scaled Bessel functions
    return i1e(x) * scale * ex / nrm, i1e(x) * scale * ey / nrm, i0e(x) * scale

def expected_velocity(a, d, lam):
    Ox, Oy, W = operators(d, lam); rho = np.outer(a, a)
    return np.array([np.trace(rho @ Ox), np.trace(rho @ Oy)]) / np.trace(rho @ W)

EXAMPLE = [("A", (3, 1), 10), ("A", (4, -1), 5), ("A", (2, 3), 20), ("B", (3, -3), -10), ("B", (5, 0), 0), ("B", (-1, 4), 15)]

def one_decision(p, phase=True, own_kind="A", own_heading=0.0, partners=EXAMPLE):
    # perceived states of one agent: (name, direction in degrees, amplitude before normalisation)
    st = [("own heading", own_heading, 1.0)]
    st += [(f"heading of partner {q + 1}", h, np.sqrt(p["w"])) for q, (_, _, h) in enumerate(partners)]
    st += [(f"bearing of partner {q + 1} ({k})", np.degrees(np.arctan2(xy[1], xy[0])),
            np.sqrt(1 - p["w"]) * (-1.0 if (phase and k != own_kind) else 1.0)) for q, (k, xy, _) in enumerate(partners)]
    d = np.radians([s[1] for s in st]); a = np.array([s[2] for s in st]); a /= np.linalg.norm(a)
    theta = np.linspace(0, 2 * np.pi, p["n_dir"], endpoint=False)
    amp = a[:, None] * np.exp(0.5 * p["lam"] * (np.cos(theta[None, :] - d[:, None]) - 1))
    prob = amp.sum(0) ** 2; prob /= prob.sum()
    return dict(names=[s[0] for s in st], d=d, a=a, theta=theta, prob=prob,
                v_direct=np.array([prob @ np.cos(theta), prob @ np.sin(theta)]), v_operator=expected_velocity(a, d, p["lam"]))

ex, ex0 = one_decision(BASE, phase=True), one_decision(BASE, phase=False)
table_example = pd.DataFrame({"perceived state": ex["names"], "direction (deg)": np.round(np.degrees(ex["d"]), 1), "amplitude": np.round(ex["a"], 3)})
table_example.to_csv(os.path.join(OUT, "table_worked_example.csv"), index=False)
display(table_example)
deg = np.degrees(ex["theta"]); deg = np.where(deg > 180, deg - 360, deg)
def mass(e, lo, hi): return float(e["prob"][(deg >= lo) & (deg <= hi)].sum())
def heading(v): return float(np.degrees(np.arctan2(v[1], v[0])))
example_lines = [
    f"Most probable heading: {deg[np.argmax(ex['prob'])]:.0f} deg (probability {ex['prob'].max():.3f} on the grid of {BASE['n_dir']} headings)",
    f"Probability of a heading between -6 and 21 deg: {mass(ex, -6, 21):.2f} with the identity phase, {mass(ex0, -6, 21):.2f} without",
    f"Probability of turning toward the two B partners (-60 to -9 deg): {mass(ex, -60, -9):.3f} with the identity phase, {mass(ex0, -60, -9):.3f} without",
    f"Expected heading: {heading(ex['v_operator']):.1f} deg with the identity phase, {heading(ex0['v_operator']):.1f} deg without",
    f"Expected velocity / v0, operator form: ({ex['v_operator'][0]:.6f}, {ex['v_operator'][1]:.6f})",
    f"Expected velocity / v0, direct sum:    ({ex['v_direct'][0]:.6f}, {ex['v_direct'][1]:.6f})"]
rng_ = np.random.default_rng(1); worst = 0.0; th_ = ex["theta"]
for _ in range(2000):                                         # 2,000 random sets of perceived states with random phases
    d_ = rng_.uniform(0, 2 * np.pi, 13); a_ = np.sqrt(rng_.uniform(0.1, 1, 13)) * rng_.choice([1, -1], 13, p=[0.8, 0.2]); a_ /= np.linalg.norm(a_)
    pr = ((a_[:, None] * np.exp(0.5 * BASE["lam"] * (np.cos(th_[None, :] - d_[:, None]) - 1))).sum(0)) ** 2; pr /= pr.sum()
    worst = max(worst, np.abs(expected_velocity(a_, d_, BASE["lam"]) - np.array([pr @ np.cos(th_), pr @ np.sin(th_)])).max())
example_lines.append(f"Largest difference between operator form and direct sum over 2,000 random sets: {worst:.1e}")
print("\n".join(example_lines))

In [ ]:
# Figure 1: one decision. (a) what the agent sees and whom it picks, (b) the perceived states,
# (c) their sum squared = heading probabilities, and one heading drawn from them.
from matplotlib.patches import FancyBboxPatch, Wedge
BLUE, ORANGE = KIND_COLORS[0], KIND_COLORS[1]
LBLUE, LORANGE, HEAD = "#b9d3f1", "#f6c7ae", "#a3a29b"
fine = np.linspace(-180, 180, 7201)
def amplitudes(e):
    return e["a"][:, None] * np.exp(0.5 * BASE["lam"] * (np.cos(np.radians(fine)[None, :] - e["d"][:, None]) - 1))
def density(e):
    q = amplitudes(e).sum(0) ** 2
    return q / (q.sum() * (fine[1] - fine[0])) * (360 / BASE["n_dir"])          # probability per grid step

fig = plt.figure(figsize=(7.4, 3.1))
gs = fig.add_gridspec(1, 3, width_ratios=[1.12, 1, 1], wspace=0.40, left=0.03, right=0.99, top=0.88, bottom=0.16)

# ---- (a) physical space: visible individuals, the six partners, their headings and bearings
a0 = fig.add_subplot(gs[0, 0])
a0.add_patch(Wedge((0, 0), 6.9, -135, 135, facecolor="#f0efec", edgecolor="none"))
a0.text(-3.7, 0.0, "blind\nzone", color=INK2, fontsize=7.5, ha="center", va="center")
others = [((1.4, 1.6), "A"), ((1.8, -1.9), "B"), ((-0.8, 2.0), "B"), ((4.9, 2.6), "B"), ((1.1, -4.6), "A"), ((-1.6, -2.6), "A"),
          ((5.6, -2.9), "A"), ((2.6, 5.4), "B"), ((-3.0, 4.1), "A"), ((6.3, 0.9), "B")]
for xy, k in others:
    a0.plot(*xy, "o", ms=4.2, mfc="white", mec=BLUE if k == "A" else ORANGE, mew=1.0, zorder=2)
for k, xy, h in EXAMPLE:
    col = BLUE if k == "A" else ORANGE
    a0.plot([0, xy[0]], [0, xy[1]], color=col, lw=1.0, ls="-" if k == "A" else (0, (3, 2)), zorder=1)
    a0.annotate("", xy=(xy[0] + 1.35 * np.cos(np.radians(h)), xy[1] + 1.35 * np.sin(np.radians(h))), xytext=xy,
                arrowprops=dict(arrowstyle="-|>", color=col, lw=1.6, mutation_scale=9), zorder=3)
    a0.plot(*xy, "o", color=col, ms=5, zorder=4, mec="white", mew=0.7)
a0.annotate("", xy=(1.5, 0), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=INK, lw=2.0, mutation_scale=11), zorder=5)
a0.plot(0, 0, "o", color=INK, ms=6, zorder=6, mec="white", mew=0.7)
a0.text(-0.3, -0.95, "agent", fontsize=7.5, color=INK, ha="center", va="center", zorder=6)
a0.legend([plt.Line2D([], [], color=BLUE, lw=1.2, marker="o", ms=4.5),
           plt.Line2D([], [], color=ORANGE, lw=1.2, ls=(0, (3, 2)), marker="o", ms=4.5),
           plt.Line2D([], [], color=INK2, lw=0, marker="o", ms=4.2, mfc="white", mew=1.0)],
          ["partner, own kind (+)", "partner, other kind (−)", "seen, not chosen"],
          loc="upper center", bbox_to_anchor=(0.5, 0.08), fontsize=7.5, handlelength=2.4, borderaxespad=0, ncol=1)
a0.set_xlim(-5.6, 7.4); a0.set_ylim(-7.2, 7.2); a0.set_aspect("equal"); a0.axis("off")
a0.set_title("(a)  what the agent sees", loc="left", fontsize=9.5, color=INK, y=1.0 + 0.0)

# ---- (b) perceived states
a1 = fig.add_subplot(gs[0, 1]); amp = amplitudes(ex)
a1.axhline(0, color=GRID, lw=0.8)
for name, y in zip(ex["names"], amp):
    if "heading" in name: a1.plot(fine, np.where(np.abs(y) > 2e-3, y, np.nan), color=HEAD, lw=0.9)
for name, y in zip(ex["names"], amp):
    y = np.where(np.abs(y) > 2e-3, y, np.nan)
    if "(A)" in name: a1.plot(fine, y, color=BLUE, lw=1.7)
    if "(B)" in name: a1.plot(fine, y, color=ORANGE, lw=1.7, ls=(0, (3, 2)))
a1.set_xlim(-90, 125); a1.set_xticks([-60, 0, 60, 120]); a1.set_ylim(-0.24, 0.5)
a1.set_xlabel(r"heading $\theta$ (degrees)"); a1.set_ylabel("amplitude")
a1.text(-4, 0.44, "headings", fontsize=7.5, color=INK2, ha="center")
a1.text(68, 0.19, "bearings,\nown kind", fontsize=7.5, color=BLUE, ha="left")
a1.text(52, -0.2, "bearings,\nother kind", fontsize=7.5, color=ORANGE, ha="center", va="bottom")
a1.text(100, -0.215, "", fontsize=7)
a1.set_title("(b)  perceived states", loc="left", fontsize=9.5, color=INK)

# ---- (c) sum, squared: heading probabilities
a2 = fig.add_subplot(gs[0, 2]); p1 = density(ex)
psi = amp.sum(0); psi = psi / np.abs(psi).max() * p1.max()
a2.fill_between(fine, 0, p1, color="#d6d4cc", lw=0)
a2.plot(fine, p1, color=INK, lw=1.8)
a2.axhline(0, color=GRID, lw=0.8)
a2.set_xlim(-90, 125); a2.set_xticks([-60, 0, 60, 120]); a2.set_ylim(0, 1.12 * p1.max())
a2.set_xlabel(r"heading $\theta$ (degrees)"); a2.set_ylabel(f"probability per {360 // BASE['n_dir']}" + r"$^\circ$")
a2.text(24, 0.90 * p1.max(), r"$|\psi(\theta)|^2$," + "\n" + r"$\psi$ = sum of (b)", fontsize=8, color=INK, va="top")
drawn = 12.0                                              # one heading drawn at random from |psi|^2
a2.annotate("heading drawn\nat random", xy=(drawn, 0.0), xytext=(42, 0.03), fontsize=7.2, color=INK, ha="left",
            arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.0, mutation_scale=8))
a2.plot([drawn, drawn], [0, float(np.interp(drawn, fine, p1))], color=INK, lw=1.2, ls=(0, (2, 1.5)))
a2.set_title("(c)  sum, then square", loc="left", fontsize=9.5, color=INK)

for a in (a1, a2):
    a.tick_params(labelsize=7.5); a.xaxis.label.set_size(8); a.yaxis.label.set_size(8)
ytop = a1.get_position().y1 + 0.012
for a, t in [(a0, "(a)  partners"), (a1, "(b)  perceived as amplitudes"), (a2, "(c)  sum, square, draw")]:
    a.set_title("", loc="left"); fig.text(a.get_position().x0 if a is not a0 else 0.035, ytop, t, fontsize=9.5, color=INK, va="bottom")
fig.savefig(os.path.join(OUT, "fig1_framework.png"), dpi=300); plt.show()


## 15. Vicsek-like alignment as a limit

Without identity and with perceived headings only (own heading, amplitude 1; six partners' headings, amplitude $\sqrt{w}$), the direction of the expected velocity is compared with the direction of the weighted mean heading $\sum_m a_m\hat e(d_m)$, which is the Vicsek rule. The partners' headings are drawn from a normal distribution around the agent's own heading; 5,000 draws per row.

In [ ]:
def vicsek_gap(spread_deg, lam, w, draws=5000, seed=0):
    rng_ = np.random.default_rng(seed)
    a = np.r_[1.0, np.full(6, np.sqrt(w))]; a /= np.linalg.norm(a)
    gap = []
    for _ in range(draws):
        d = np.r_[0.0, rng_.normal(0, np.radians(spread_deg), 6)]
        v = expected_velocity(a, d, lam); u = np.array([a @ np.cos(d), a @ np.sin(d)])
        gap.append(np.degrees(np.arccos(np.clip(v @ u / np.linalg.norm(v) / np.linalg.norm(u), -1, 1))))
    return np.mean(gap), np.percentile(gap, 95)

rows = []
for spread in [5, 10, 20, 30, 45, 60, 90]:
    m1, q1 = vicsek_gap(spread, BASE["lam"], BASE["w"]); m2, q2 = vicsek_gap(spread, 0.5, BASE["w"])
    rows.append({"spread of headings (deg)": spread, f"mean angle, lambda = {BASE['lam']:.0f}": f"{m1:.2f}",
                 f"95th percentile, lambda = {BASE['lam']:.0f}": f"{q1:.2f}", "mean angle, lambda = 0.5": f"{m2:.3f}",
                 "95th percentile, lambda = 0.5": f"{q2:.3f}"})
table_vicsek = pd.DataFrame(rows); table_vicsek.to_csv(os.path.join(OUT, "table_vicsek_limit.csv"), index=False)
print("Angle (degrees) between the expected heading without identity and the weighted mean heading.")
display(table_vicsek)

## 16. Numbers for the manuscript

Every number the text needs, in one file: `output/numbers_for_manuscript.txt`.

In [ ]:
lines = ["=" * 110,
         f"Settings: N = {BASE['N']}, G = {G}, T = {T_RUN}, delta0 = {BASE['delta0']}, R = {BASE['R']}, lambda = {BASE['lam']}, one group: C > {C_ONE}, "
         f"contact link = {CONTACT} body lengths, quick test = {QUICK_TEST}",
         f"Mixing entropy of randomly placed kinds: {M_RANDOM:.2f}",
         f"Width of the starting disc: " + ", ".join(f"N = {N_}: {start_width(N_):.1f}" for N_ in LIMIT_PLAN),
         "\nTable 2 (model and controls):", table1.to_string(index=False), spacing_line, sorting_time_line,
         "\nTests on the mixing entropy:", tests.to_string(index=False),
         "\nTable 3 (decision rules):", table2.to_string(index=False),
         "\nSight range (flock width / sight range):", table_limit.to_string(index=False), sight_line,
         "\nAttention to other kinds (delta0):", table_attention.to_string(index=False),
         "\nNoise of one decision:", table_noise_map.to_string(index=False),
         "\nNoise sweep:", table_noise.to_string(index=False),
         "\nSensitivity:", table_sens.to_string(index=False),
         "\nCue controls:", table_cues.to_string(index=False),
         "\nLarge flock followed for longer:", table_big.to_string(index=False),
         "\nPerception strength:", table_strength.to_string(index=False),
         "\nLong runs:", table_long.to_string(index=False), trend_line,
         "\nWorked example:", table_example.to_string(index=False), "\n".join(example_lines),
         "\nVicsek limit:", table_vicsek.to_string(index=False),
         "\nFiles: " + ", ".join(sorted(f for f in os.listdir(OUT) if f != "cache")), "=" * 110]
text = "\n".join(lines)
with open(os.path.join(OUT, "numbers_for_manuscript.txt"), "w", encoding="utf-8") as fh:
    fh.write(text)
print(text)

## 17. Everything for the paper in one file

This cell saves the data behind the figures and packs all results into **one file: `output/paper_data.zip`**.

| File | Content |
|---|---|
| `all_runs.csv` | one line per run, for every section: case, seed, parameters changed, and the final $M$, $C$, $\Phi$, $\mathcal S$, distances between kinds, groups by contact, nearest-neighbour distance |
| `time_series.npz` | the curves over time of every run of Tables 2 and 3, of the long runs and of the large flock |
| `final_positions_fig2.npz` | positions, headings and kinds of the three flocks of Figure 2 |
| `paper_data.zip` | all tables, figures, `numbers_for_manuscript.txt` and the three files above (not the video, not the saved runs) |

In [ ]:
import zipfile

all_runs = pd.DataFrame(ALL_RUNS).drop_duplicates(subset=["section", "case", "seed", "changes", "steps"]).reset_index(drop=True)
all_runs.to_csv(os.path.join(OUT, "all_runs.csv"), index=False)

series = {}
def add_series(tag, runs, keys):
    series[f"{tag}__t"] = runs[0]["rec"]["t"]
    for k in keys:
        series[f"{tag}__{k}"] = np.array([r["rec"][k] for r in runs])
for c in TABLE1:
    add_series(f"table1__{c}", R[c], ["M", "C", "Phi", "S", "gap_min", "gap_max"])
for c in ["decohered", "vector"]:
    add_series(f"table2__{c}", [rules[(c, s_, (), T_RUN)] for s_ in MAIN_SEEDS], ["M", "C", "Phi"])
add_series("long_runs", LR, ["M", "C", "Phi", "gap_min", "gap_max"])
for q, js in big_jobs.items():
    add_series(f"large_flock__width_over_R_{q}", [big[j] for j in js], ["M", "C", "Phi", "gap_min", "gap_max"])
np.savez_compressed(os.path.join(OUT, "time_series.npz"), **series)

shown = {}
for c in SHOW:
    X, th, kd = R[c][show]["final"]
    shown[f"{c}__positions"], shown[f"{c}__headings"], shown[f"{c}__kinds"] = X, th, kd
np.savez_compressed(os.path.join(OUT, "final_positions_fig2.npz"), **shown)

zip_path = os.path.join(OUT, "paper_data.zip")
packed = sorted(f for f in os.listdir(OUT) if f.endswith((".csv", ".png", ".txt", ".npz")))
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in packed:
        z.write(os.path.join(OUT, f), f)
print(f"all_runs.csv: {len(all_runs)} runs")
print(f"{zip_path}: {len(packed)} files, {os.path.getsize(zip_path) / 1e6:.1f} MB")
print("\n".join("   " + f for f in packed))
print("\n>>> Send this one file:", zip_path)

## 18. Video — the three cases side by side

One run (`SHOW_SEED`). The camera follows the flock and frames all agents whenever they fit; if a flock splits it follows the largest group. Saved as MP4 if `ffmpeg` is available (system, or `pip install imageio-ffmpeg`), otherwise as an animated GIF.

In [ ]:
def find_ffmpeg():
    if shutil.which("ffmpeg"):
        return shutil.which("ffmpeg")
    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return None

def camera(frames, shrink=0.1):
    # the window follows the flock, widens at once when needed and narrows slowly
    path, half = [], None
    for X, _ in frames:
        ctr, h, shown = view(X, BASE["R"])
        half = h if (half is None or h > half) else (1 - shrink) * half + shrink * h
        path.append((ctr.copy(), half, shown))
    return path

print("Simulating the video runs ...", flush=True)
try:
    from joblib import Parallel, delayed
    vids = Parallel(n_jobs=min(len(SHOW), os.cpu_count()))(delayed(run)(c, SHOW_SEED, dict(BASE), T_RUN, EVERY, VIDEO_EVERY) for c in SHOW)
except ImportError:
    vids = [run(c, SHOW_SEED, dict(BASE), T_RUN, EVERY, VIDEO_EVERY) for c in SHOW]

fig, axes = plt.subplots(1, len(SHOW), figsize=(4.2 * len(SHOW), 4.7), squeeze=False)
arts = []
for a_, c, v in zip(axes[0], SHOW, vids):
    X0, th0 = v["frames"][0]; kd = v["final"][2]
    q = a_.quiver(X0[:, 0], X0[:, 1], np.cos(th0), np.sin(th0), color=[KIND_COLORS[k % len(KIND_COLORS)] for k in kd],
                  angles="xy", scale_units="xy", scale=0.5, width=0.005)
    a_.set_aspect("equal"); a_.set_xticks([]); a_.set_yticks([])
    for sp in a_.spines.values():
        sp.set_visible(True); sp.set_color("#c3c2b7")
    a_.set_title(LABEL[c], loc="left", fontsize=10, color=INK)
    arts.append((q, a_.text(0.02, 0.97, "", transform=a_.transAxes, va="top", fontsize=9, color=INK), v["frames"], camera(v["frames"])))
fig.tight_layout()

def update(f):
    for q, txt, frames, cam in arts:
        X, th = frames[f]; q.set_offsets(X); q.set_UVC(np.cos(th), np.sin(th))
        (cx, cy), h, shown = cam[f]
        q.axes.set_xlim(cx - h, cx + h); q.axes.set_ylim(cy - h, cy + h)
        txt.set_text(f"step {VIDEO_EVERY * f}" + ("" if shown > 0.995 else f"   ({100 * shown:.0f}% in view)"))
    return []

anim = animation.FuncAnimation(fig, update, frames=len(arts[0][2]), blit=False)
ff = find_ffmpeg()
if ff:
    plt.rcParams["animation.ffmpeg_path"] = ff
    video_path = os.path.join(OUT, "video_three_cases.mp4")
    anim.save(video_path, writer=animation.FFMpegWriter(fps=VIDEO_FPS, bitrate=3000), dpi=110); plt.close(fig)
    print("Saved", video_path); display(Video(video_path, embed=True, width=900))
else:
    video_path = os.path.join(OUT, "video_three_cases.gif")
    anim.save(video_path, writer=animation.PillowWriter(fps=VIDEO_FPS), dpi=70); plt.close(fig)
    print("ffmpeg not found, saved a GIF:", video_path); display(Image(filename=video_path))